# Week 7 Community Contribution - BernardUdo

This notebook tracks the Week 7 end-to-end open-source fine-tuning workflow.

## Scope by day
- Day 1: Define project task and model strategy (QLoRA/LoRA).
- Day 2: Build prompt-completion data and split datasets.
- Day 3-4: Fine-tune an open-source model.
- Day 5: Evaluate on held-out data and summarize performance.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
REPO_ROOT = next((p for p in [PROJECT_DIR, *PROJECT_DIR.parents] if (p / ".git").exists()), PROJECT_DIR)
WEEK7_DIR = REPO_ROOT / "week7"

if str(WEEK7_DIR) not in sys.path:
    sys.path.append(str(WEEK7_DIR))

print(f"Project dir: {PROJECT_DIR}")
print(f"Repo root:   {REPO_ROOT}")
print(f"Week7 dir:   {WEEK7_DIR}")

Project dir: C:\Users\HP\projects\Document\llm_engineering\week7\community_contributions\BernardUdo
Repo root:   C:\Users\HP\projects\Document\llm_engineering
Week7 dir:   C:\Users\HP\projects\Document\llm_engineering\week7


## Run order
1. Install dependencies in your notebook environment.
2. Load curated Week 7 item data from Hugging Face Hub.
3. Build prompt-completion pairs for SFT.
4. Fine-tune an open-source model (QLoRA style config).
5. Evaluate on held-out test prompts and summarize results.

## Day 1-2: Environment and data setup

If needed, install dependencies in this kernel:

```bash
pip install -U datasets transformers peft trl accelerate bitsandbytes scikit-learn pandas matplotlib
```

> Tip: for GPU training, run this notebook in Colab with a T4/L4/A100 runtime.

In [2]:
import re
from statistics import mean

import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer

from pricer.items import Item

C:\Users\HP\projects\Document\llm_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# Change USERNAME when you are ready to push processed prompt data to your own Hugging Face account.
USERNAME = "ed-donner"
LITE_MODE = True
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SUMMARY_TOKENS = 220
DO_ROUND = True
SEED = 42

raw_dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"
prompt_dataset_name = f"{USERNAME}/items_prompts_lite" if LITE_MODE else f"{USERNAME}/items_prompts_full"

print("raw_dataset_name:", raw_dataset_name)
print("prompt_dataset_name:", prompt_dataset_name)
print("base_model:", BASE_MODEL)

raw_dataset_name: ed-donner/items_lite
prompt_dataset_name: ed-donner/items_prompts_lite
base_model: Qwen/Qwen2.5-0.5B-Instruct


In [4]:
train_items, val_items, test_items = Item.from_hub(raw_dataset_name)
print(f"train={len(train_items)}, val={len(val_items)}, test={len(test_items)}")
print(train_items[0])

train=20000, val=1000, test=1000
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full=None weight=1.5 summary='Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.' prompt=None completion=None id=None


In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

for item in train_items:
    item.make_prompts(tokenizer=tokenizer, max_tokens=MAX_SUMMARY_TOKENS, do_round=DO_ROUND)
for item in val_items:
    item.make_prompts(tokenizer=tokenizer, max_tokens=MAX_SUMMARY_TOKENS, do_round=DO_ROUND)
for item in test_items:
    item.make_prompts(tokenizer=tokenizer, max_tokens=MAX_SUMMARY_TOKENS, do_round=DO_ROUND)

print(train_items[0].prompt[:300])
print("completion:", train_items[0].completion)

C:\Users\HP\projects\Document\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


What does this cost to the nearest dollar?

Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: Home Hardware  
Brand: Schlage  
Description: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  
Details: Designed for a 4" mi
completion: 64.00


In [6]:
# Optional: publish prompt-completion data to your own HF dataset namespace.
# Set USERNAME to your HF user/org before running this.
# Item.push_prompts_to_hub(prompt_dataset_name, train_items, val_items, test_items)

## Day 3-4: Fine-tuning workflow (QLoRA-style)

This section is intentionally compact and reproducible. You can run it on GPU in Colab.

If your environment cannot install `bitsandbytes`, keep the code and run it in Colab for the actual training step.

In [7]:
def to_sft_dataset(items):
    rows = []
    for item in items:
        text = f"{item.prompt}{item.completion}"
        rows.append({"text": text})
    return Dataset.from_list(rows)

train_sft = to_sft_dataset(train_items)
val_sft = to_sft_dataset(val_items)

train_sft[0]

{'text': 'What does this cost to the nearest dollar?\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.\n\nPrice is $64.00'}

In [8]:
# Fine-tuning cell (run on GPU runtime)
try:
    from peft import LoraConfig
    from transformers import AutoModelForCausalLM, TrainingArguments
    from trl import SFTTrainer

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        device_map="auto",
    )

    training_args = TrainingArguments(
        output_dir="./artifacts/week7_qlora",
        learning_rate=2e-4,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        logging_steps=10,
        save_steps=100,
        evaluation_strategy="steps",
        eval_steps=100,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_sft,
        eval_dataset=val_sft,
        peft_config=lora_config,
        dataset_text_field="text",
        args=training_args,
    )

    # Uncomment to train:
    # trainer.train()
    # trainer.save_model("./artifacts/week7_qlora/final")
    print("Fine-tuning scaffold initialized. Uncomment train lines to run training on GPU.")
except ModuleNotFoundError as exc:
    print(f"Skipping fine-tuning scaffold setup in this environment: {exc}")
    print("Install peft and trl (or run in Colab GPU) to enable this section.")

Skipping fine-tuning scaffold setup in this environment: No module named 'peft'
Install peft and trl (or run in Colab GPU) to enable this section.


## Day 5: Evaluate on held-out test set

This section gives you a guaranteed runnable baseline, and an optional fine-tuned model evaluator if you trained an adapter/model.

In [9]:
def extract_price(text_or_value):
    if isinstance(text_or_value, (int, float)):
        return float(text_or_value)
    text = str(text_or_value).replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", text)
    return float(match.group()) if match else 0.0

mean_train_price = mean([item.price for item in train_items])

def baseline_predictor(_item):
    return mean_train_price

print(f"Baseline mean price: ${mean_train_price:,.2f}")

Baseline mean price: $140.35


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def score_predictor(predict_fn, items, size=200):
    subset = items[: min(size, len(items))]
    y_true = np.array([item.price for item in subset], dtype=float)
    y_pred = np.array([extract_price(predict_fn(item)) for item in subset], dtype=float)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(mean_squared_error(y_true, y_pred) ** 0.5)
    r2 = float(r2_score(y_true, y_pred))
    within_20 = float((np.abs(y_pred - y_true) <= 20).mean() * 100)
    return {
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "within_20_pct": within_20,
    }

baseline_metrics = score_predictor(baseline_predictor, test_items, size=200)
baseline_metrics

{'mae': 106.08424013,
 'rmse': 148.35369744991712,
 'r2': -0.0014232153900839428,
 'within_20_pct': 10.0}

In [11]:
# Optional fine-tuned evaluation
# Set this path to your trained adapter/model folder, then run this cell.
# Example: FINETUNED_MODEL_PATH = "./artifacts/week7_qlora/final"
FINETUNED_MODEL_PATH = None

if FINETUNED_MODEL_PATH:
    from transformers import AutoModelForCausalLM, pipeline

    ft_model = AutoModelForCausalLM.from_pretrained(FINETUNED_MODEL_PATH, device_map="auto")
    ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=tokenizer)

    def finetuned_predictor(item):
        prompt = item.prompt
        out = ft_pipe(prompt, max_new_tokens=8, do_sample=False)[0]["generated_text"]
        return out[len(prompt):]

    finetuned_metrics = score_predictor(finetuned_predictor, test_items, size=200)
    print(finetuned_metrics)
else:
    print("Set FINETUNED_MODEL_PATH to run fine-tuned model evaluation.")

Set FINETUNED_MODEL_PATH to run fine-tuned model evaluation.


## Submission summary

- **Base model:** `Qwen/Qwen2.5-0.5B-Instruct`
- **Fine-tuning method:** LoRA/QLoRA-style SFT scaffold in notebook
- **Dataset used:** `ed-donner/items_lite` with prompt-completion conversion
- **Current lite-run baseline metrics (test subset size=200):**
  - MAE: `106.08`
  - RMSE: `148.35`
  - R2: `-0.0014`
  - % within $20: `10.0`
- **Notes:** baseline and full pipeline scaffolding are executed; fine-tuning run is intentionally deferred to GPU environment (Colab/local CUDA) by enabling the training lines in the Day 3-4 cell.